## 1. Import

In [44]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
from flaml import AutoML

## 2. 데이터 전처리

In [45]:
train = pd.read_csv("C:/Users/Konyang/Downloads/ACD2-Week12-1/dataset/train.csv")

# year, month, item_id 기준으로 value 합산 (seq만 다르다면 value 합산)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)["value"]
    .sum()
)

# year, month를 하나의 키(ym)로 묶기
monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# item_id × ym 피벗 (월별 총 무역량 매트릭스 생성)
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

pivot.head()

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0


## 3. 공행성쌍 탐색
- 각 (A, B) 쌍에 대해 lag = 1 ~ max_lag까지 Pearson 상관계수 계산
- 절댓값이 가장 큰 상관계수와 lag를 선택
- |corr| >= corr_threshold이면 A→B 공행성 있다고 판단

In [46]:
def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def find_comovement_pairs(pivot, max_lag=10, min_nonzero=12, corr_threshold=0.2):
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            # lag = 1 ~ max_lag 탐색
            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue
                corr = safe_corr(x[:-lag], y[lag:])
                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            # 임계값 이상이면 공행성쌍으로 채택
            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    pairs = pd.DataFrame(results)
    return pairs

pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()

100it [00:10,  9.38it/s]

탐색된 공행성쌍 수: 7386


,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,AHMDUILJ,10,-0.243856
1,AANGBULD,APQGTRMF,8,-0.480115
2,AANGBULD,AXULOHBQ,1,-0.255074
3,AANGBULD,BEZYMBBT,10,-0.483240
4,AANGBULD,BLANHGYY,10,0.252586


## 4. 회귀 모델 학습
- 시계열 데이터 안에서 '한 달 뒤 총 무역량(value)을 맞추는 문제'로 self-supervised 학습
- 탐색된 모든 공행성쌍 (A,B)에 대해 월 t마다 학습 샘플 생성
- input X:
1) B_t (현재 총 무역량(value))
2) B_{t-1} (직전 달 총 무역량(value))
3) A_{t-lag} (lag 반영된 총 무역량(value))
4) max_corr, best_lag (관계 특성)
- target y:
1) B_{t+1} (다음 달 총 무역량(value))
- 이러한 모든 샘플을 합쳐 LinearRegression 회귀 모델을 학습

In [47]:
def build_training_data(pivot, pairs):
    """
    공행성쌍 + 시계열을 이용해 (X, y) 학습 데이터를 만드는 함수
    input X:
      - b_t, b_t_1, a_t_lag, max_corr, best_lag
    target y:
      - b_t_plus_1
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t+1이 존재하고, t-lag >= 0인 구간만 학습에 사용
        for t in range(max(lag, 1), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]

            rows.append({
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                "target": b_t_plus_1,
            })

    df_train = pd.DataFrame(rows)
    return df_train

df_train_model = build_training_data(pivot, pairs)
print('생성된 학습 데이터의 shape :', df_train_model.shape)
df_train_model.head()

생성된 학습 데이터의 shape : (267374, 6)


,b_t,b_t_1,a_t_lag,max_corr,best_lag,target
0,141264.0,89292.0,14276.0,-0.243856,10.0,71149.0
1,71149.0,141264.0,52347.0,-0.243856,10.0,68964.0
2,68964.0,71149.0,53549.0,-0.243856,10.0,105738.0
3,105738.0,68964.0,0.0,-0.243856,10.0,146741.0
4,146741.0,105738.0,26997.0,-0.243856,10.0,81879.0


In [6]:
# 회귀모델 학습
#feature_cols = ['b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag']

#train_X = df_train_model[feature_cols].values
#train_y = df_train_model["target"].values

#reg = LinearRegression()
#reg.fit(train_X, train_y)

In [48]:
from flaml import AutoML
feature_cols = ['b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag']
train_X = df_train_model[feature_cols].values
train_y = df_train_model["target"].values
automl = AutoML()

settings = {
    "time_budget": 60,
    "task": "regression",
    "metric" : "mse",
    "estimator_list" : ["rf", "lgbm", "xgboost"],
    "log_file_name": "automl_multi.log",
}

automl.fit(
    X_train=train_X,
    y_train=train_y,
    **settings
)
reg = automl

[flaml.automl.logger: 11-14 11:44:09] {1752} INFO - task = regression
[flaml.automl.logger: 11-14 11:44:09] {1763} INFO - Evaluation method: holdout
[flaml.automl.logger: 11-14 11:44:09] {1862} INFO - Minimizing error metric: mse
[flaml.automl.logger: 11-14 11:44:09] {1979} INFO - List of ML learners in AutoML Run: ['rf', 'lgbm', 'xgboost']
[flaml.automl.logger: 11-14 11:44:09] {2282} INFO - iteration 0, current learner rf
[flaml.automl.logger: 11-14 11:44:09] {2417} INFO - Estimated sufficient time budget=26323s. Estimated necessary time budget=26s.
[flaml.automl.logger: 11-14 11:44:09] {2466} INFO -  at 0.2s,	estimator rf's best error=22152505392384.7344,	best estimator rf's best error=22152505392384.7344
[flaml.automl.logger: 11-14 11:44:09] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 11-14 11:44:09] {2466} INFO -  at 0.2s,	estimator lgbm's best error=96946371152618.4375,	best estimator rf's best error=22152505392384.7344
[flaml.automl.logger: 11-14 11:44:0

C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[flaml.automl.logger: 11-14 11:44:09] {2466} INFO -  at 0.4s,	estimator rf's best error=13114665205838.8281,	best estimator rf's best error=13114665205838.8281
[flaml.automl.logger: 11-14 11:44:09] {2282} INFO - iteration 7, current learner rf
[flaml.automl.logger: 11-14 11:44:09] {2466} INFO -  at 0.6s,	estimator rf's best error=8895098540897.5742,	best estimator rf's best error=8895098540897.5742
[flaml.automl.logger: 11-14 11:44:09] {2282} INFO - iteration 8, current learner lgbm
[flaml.automl.logger: 11-14 11:44:09] {2466} INFO -  at 0.6s,	estimator lgbm's best error=14200111236799.2461,	best estimator rf's best error=8895098540897.5742
[flaml.automl.logger: 11-14 11:44:09] {2282} INFO - iteration 9, current learner lgbm
[flaml.automl.logger: 11-14 11:44:09] {2466} INFO -  at 0.6s,	estimator lgbm's best error=13298279865313.4238,	best estimator rf's best error=8895098540897.5742
[flaml.automl.logger: 11-14 11:44:09] {2282} INFO - iteration 10, current learner xgboost
[flaml.automl.

C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[flaml.automl.logger: 11-14 11:44:10] {2466} INFO -  at 0.9s,	estimator rf's best error=8542028399466.6758,	best estimator rf's best error=8542028399466.6758
[flaml.automl.logger: 11-14 11:44:10] {2282} INFO - iteration 18, current learner xgboost
[flaml.automl.logger: 11-14 11:44:10] {2466} INFO -  at 1.0s,	estimator xgboost's best error=15632847087461.5312,	best estimator rf's best error=8542028399466.6758
[flaml.automl.logger: 11-14 11:44:10] {2282} INFO - iteration 19, current learner rf
[flaml.automl.logger: 11-14 11:44:10] {2466} INFO -  at 1.1s,	estimator rf's best error=8542028399466.6758,	best estimator rf's best error=8542028399466.6758
[flaml.automl.logger: 11-14 11:44:10] {2282} INFO - iteration 20, current learner rf
[flaml.automl.logger: 11-14 11:44:10] {2466} INFO -  at 1.4s,	estimator rf's best error=8310997252812.4775,	best estimator rf's best error=8310997252812.4775
[flaml.automl.logger: 11-14 11:44:10] {2282} INFO - iteration 21, current learner rf
[flaml.automl.log

...

[flaml.automl.logger: 11-14 11:44:51] {2466} INFO -  at 41.8s,	estimator lgbm's best error=104840550139.5695,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:51] {2282} INFO - iteration 95, current learner rf
[flaml.automl.logger: 11-14 11:44:52] {2466} INFO -  at 43.7s,	estimator rf's best error=208892249548.1555,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:52] {2282} INFO - iteration 96, current learner rf
[flaml.automl.logger: 11-14 11:44:53] {2466} INFO -  at 44.1s,	estimator rf's best error=208892249548.1555,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:53] {2282} INFO - iteration 97, current learner rf
[flaml.automl.logger: 11-14 11:44:54] {2466} INFO -  at 45.5s,	estimator rf's best error=208892249548.1555,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:54] {2282} INFO - iteration 98, current learner rf
[flaml.automl.logger: 

C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[flaml.automl.logger: 11-14 11:44:56] {2466} INFO -  at 47.5s,	estimator rf's best error=208892249548.1555,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:56] {2282} INFO - iteration 101, current learner rf
[flaml.automl.logger: 11-14 11:44:57] {2466} INFO -  at 48.8s,	estimator rf's best error=200844630165.6815,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:57] {2282} INFO - iteration 102, current learner lgbm
[flaml.automl.logger: 11-14 11:44:58] {2466} INFO -  at 49.1s,	estimator lgbm's best error=104840550139.5695,	best estimator lgbm's best error=104840550139.5695
[flaml.automl.logger: 11-14 11:44:58] {2282} INFO - iteration 103, current learner lgbm


C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[flaml.automl.logger: 11-14 11:45:08] {2466} INFO -  at 59.0s,	estimator lgbm's best error=30931042644.3674,	best estimator lgbm's best error=30931042644.3674
[flaml.automl.logger: 11-14 11:45:08] {2282} INFO - iteration 104, current learner xgboost
[flaml.automl.logger: 11-14 11:45:08] {2466} INFO -  at 59.4s,	estimator xgboost's best error=5183864422843.8213,	best estimator lgbm's best error=30931042644.3674
[flaml.automl.logger: 11-14 11:45:08] {2282} INFO - iteration 105, current learner xgboost
[flaml.automl.logger: 11-14 11:45:09] {2466} INFO -  at 59.8s,	estimator xgboost's best error=5183864422843.8213,	best estimator lgbm's best error=30931042644.3674
[flaml.automl.logger: 11-14 11:45:19] {2724} INFO - retrain lgbm for 10.3s
[flaml.automl.logger: 11-14 11:45:19] {2727} INFO - retrained model: LGBMRegressor(colsample_bytree=0.8104448263763874,
              learning_rate=0.04185311999071249, max_bin=1023,
              min_child_samples=5, n_estimators=2305, n_jobs=-1, num_leav

In [42]:
print("=== AutoML 결과 요약 ===")
print("Best estimator (모델 타입):",  automl.best_estimator)
print("Best loss (MSE 기준):", automl.best_loss)

print("/nBest config (하이퍼파라미터):")
print(automl.best_config)

=== AutoML 결과 요약 ===
Best estimator (모델 타입): lgbm
Best loss (MSE 기준): 4713832558030.604
/nBest config (하이퍼파라미터):
{'n_estimators': 557, 'num_leaves': 16, 'min_child_samples': 2, 'learning_rate': 0.1148309446819041, 'log_max_bin': 10, 'colsample_bytree': 0.9106701277903652, 'reg_alpha': 0.0011185857949779438, 'reg_lambda': 0.0011096370527874369}


## 5. 회귀 모델 추론 및 제출(submission) 파일 생성
- 탐색된 공행성 쌍에 대해 후행 품목(following_item_id)에 대한 2025년 8월 총 무역량(value) 예측

In [49]:
def predict(pivot, pairs, reg):
    months = pivot.columns.to_list()
    n_months = len(months)

    # 가장 마지막 두 달 index (2025-7, 2025-6)
    t_last = n_months - 1
    t_prev = n_months - 2

    preds = []

    for row in tqdm(pairs.itertuples(index=False)):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t_last - lag 가 0 이상인 경우만 예측
        if t_last - lag < 0:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        a_t_lag = a_series[t_last - lag]

        X_test = np.array([[b_t, b_t_1, a_t_lag, corr, float(lag)]])
        y_pred = reg.predict(X_test)[0]

        # (후처리 1) 음수 예측 → 0으로 변환
        # (후처리 2) 소수점 → 정수 변환 (무역량은 정수 단위)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    df_pred = pd.DataFrame(preds)
    return df_pred

In [50]:
submission = predict(pivot, pairs, reg)
submission.head()

0it [00:00, ?it/s]C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Konyang\anaconda3\envs\acd2\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitte

,leading_item_id,following_item_id,value
0,AANGBULD,AHMDUILJ,242842
1,AANGBULD,APQGTRMF,244622
2,AANGBULD,AXULOHBQ,62111
3,AANGBULD,BEZYMBBT,2643540
4,AANGBULD,BLANHGYY,237264


In [51]:
submission.to_csv("C:/Users/Konyang/Downloads/ACD2-Week12-1/dataset/sample_submission.csv", index=False)